***
***
# A simple tutorial on how use the C3S to fit two-body interactions
***
***
<br>
In this tutorial we fit C3S to reference data consisting of 'Lennard-Jonesium'. 
<br><br><br><br><br>

***
***
## 1. Import modules

In [ ]:

from ase.io import read,write
from ase.build import bulk
import json
import numpy as np
import ase.db as db
from ase.visualize import view
from ase.calculators.lj import LennardJones
import matplotlib.pyplot as plt
from ccs_fit.scripts.ccs_build_db import ccs_build_db
from ccs_fit import ccs_fit

***
***
## 2. Generate training data
Curvature Constrained Splines can be fitted to a reference data-set with energies (and optionally forces) of pre-calculated structures. In this example we generate the reference data-set using a Lennard-Jones potential. We use a fcc bulk structures that we randomly rattle and scale. The results are saved to separete xyz-files (``S1.xyz``,``S2.xyz``, etc.) and stored in the directory ``CALCULATED_DATA``.


In [ ]:
LJ=bulk('Ne','fcc',a=1.55)
LJ=LJ*[3,3,3]
calc = LennardJones()
LJ.calc = calc


orig_cell = LJ.get_cell()
orig_struc = LJ.copy()

displacement_magnitude=0.02
disp_steps=5
rounds=3

trainset_list="file_list" # List of strucutres to include in the database (see below)
f = open(trainset_list, "w")
counter=1
for round in range(rounds):
    for scale in np.linspace(0.95, 1.05, 6):
        new_cell = orig_cell*scale
        new_struc = orig_struc.copy()
        new_struc.set_cell(new_cell)
        new_struc.calc = calc
        nrg = new_struc.get_potential_energy()
        for i in range(disp_steps):
            rattle_struc = new_struc.copy()
            rattle_struc.rattle(displacement_magnitude*i, seed=counter)
            rattle_struc.calc = calc
            nrg = rattle_struc.get_potential_energy()
            if nrg < 0: # We exclude structures that are unreasonably high in energy
                xyz_file=f"CALCULATED_DATA/S{counter}.xyz"
                write(xyz_file,rattle_struc)
                print(xyz_file,file=f)
                counter += 1

f.close()
       

***
***
## 3. Building a reference database

After generating the data, we collect it in an ASE database file. The ``file_list`` is a file containing a list of files to be collected into the database.

Example of a ``file_list`` file:

    CALCULATED_DATA/S1.xyz
    CALCULATED_DATA/S2.xyz
    CALCULATED_DATA/S3.xyz
    CALCULATED_DATA/S4.xyz
    
Any format supported by ASE can be read in. Note that the ``file_list`` was created alongside the xyz-files in the function above.

In [ ]:
ccs_build_db(mode="CCS",REF_DB="LJ.db",file_list="file_list",overwrite=True)

***
***
## 4. Fit training data to Curvature Constrained Splines
Finally, the splines are fitted to the target defined in the `LJ.db`  input file. The splines can be restricted to be fully repulsive ("rep"), or have a turning point/switch ("sw"), which is defined by the "Swtype" key. 


In [ ]:
### Generate input.json file

input={
    "General": {
        "Interface": "CCS",
        "FitForces": "True",
        "FitStresses": "True"
    },
    "TrainSet": "LJ.db",
    "Twobody": {
                "Ne-Ne": {
                        "Rcut": 5.0,
                        "Resolution": 0.02,
                        "SwType": "sw",
                        "ConstType" : "Mono"
                }
        }
}
#SAVE TO FILE
with open('CCS_input.json', 'w') as f:
    json.dump(input, f, indent=8)

In [ ]:
ccs_fit("CCS_input.json")

***
***
## 5. Validate your potential
Make sure your potential (at least) reproduce the data points in your training-set. Performing further tests on strucutres not included in the training set is recomended but not included in the tutorial.

In [ ]:
from ccs_fit.scripts.ccs_validate import ccs_validate
ccs_validate(mode="CCS",CCS_params="CCS_params.json",DFT_DB="LJ.db")

In [ ]:
with open("CCS_params.json", "r") as f:
    CCS_params = json.load(f)

r=np.array(CCS_params["Two_body"]["Ne-Ne"]["r"])
e=CCS_params["Two_body"]["Ne-Ne"]["spl_a"]
e_LJ= 4 * ((1 / r) ** 12 - (1 / r) ** 6)
plt.xlim(0.5,3)
plt.ylim(-1.5,1)
plt.xlabel('Distance (Å)')
plt.ylabel('Potential (eV)')
plt.plot(r,e_LJ,color='black',label="Ref. Lennard-Jones potential")
plt.plot(r,e,'--',color='red',label="Fitted potential")
plt.legend()
plt.show()

err=np.loadtxt("CCS_validate.dat")
err[:,0]=err[:,0]/err[:,3]
err[:,1]=err[:,1]/err[:,3]
plt.xlabel('Reference energy (eV/atom)')
plt.ylabel('Validation energy (eV/atom)')
plt.plot( [min(err[:,0]),max(err[:,0])],[min(err[:,0]),max(err[:,0])],'--',color='black'  )
plt.scatter(err[:,0],err[:,1],facecolors='none', edgecolors='red')
plt.show()
plt.xlabel('Reference energy (eV/atom)')
plt.ylabel('Error in fit (eV/atom)')
plt.scatter(err[:,0],err[:,2],facecolors='none', edgecolors='red')
plt.show()

try:
    err_F=np.loadtxt("CCS_error_forces.out")
    plt.xlabel('Reference force (eV/Å)')
    plt.ylabel('Fitted force (eV/Å)')
    plt.plot( [min(err_F[:,0]),max(err_F[:,0])],[min(err_F[:,0]),max(err_F[:,0])],'--',color='black')
    plt.scatter(err_F[:,0],err_F[:,1],facecolors='none', edgecolors='red',alpha=0.1 )
    plt.show()
except:
    pass

try:
    err_F=np.loadtxt("CCS_error_stresses.out")
    plt.xlabel(r'Reference stress (eV/Å$^3$)')
    plt.ylabel(r'Fitted stress (eV/Å$^3$)')
    plt.plot( [min(err_F[:,0]),max(err_F[:,0])],[min(err_F[:,0]),max(err_F[:,0])],'--',color='black')
    plt.scatter(err_F[:,0],err_F[:,1],facecolors='none', edgecolors='red',alpha=0.1 )
    plt.show()
except:
    pass


***
***
## 6. Speeding up the process
The search for inflection points often become the bottle neck in the fitting process. We can gain considerable performance by restricting this search to specific points or by limiting the search intervall. We can also force the potential to be strictly repulsive or strictly attractive by setting the `"SwType"` to `"rep"` or `"att"` instead of `"sw"`. 

In the current example we would need the potential to switch from repulsive to attractive and we therefore use the restricted search approach. 

### Using "SearchPoints"
By using the `"SearchMode" : "Point"` option we can restrict the search for inflection points to those specified in the list `"SearchPoints"`.

In [ ]:
### Generate input.json file
import json

input={
    "General": {
        "Interface": "CCS",
        "FitForces": "True",
        "FitStresses": "True"
    },
    "TrainSet": "LJ.db",
    "Twobody": {
                "Ne-Ne": {
                        "Rcut": 5.0,
                        "Resolution": 0.02,
                        "SwType": "sw",
                        "ConstType" : "Mono",
                        "SearchMode": "Point",
                        "SearchPoints": [0.5,1.0,1.5,2.0,2.5,3.0,5.0,6.0]
                }
        }
}
#SAVE TO FILE
with open('CCS_input.json', 'w') as f:
    json.dump(input, f, indent=8)

ccs_fit("CCS_input.json")

with open("CCS_params.json", "r") as f:
    CCS_params = json.load(f)

r=np.array(CCS_params["Two_body"]["Ne-Ne"]["r"])
e=CCS_params["Two_body"]["Ne-Ne"]["spl_a"]
e_LJ= 4 * ((1 / r) ** 12 - (1 / r) ** 6)
plt.xlim(0.5,3)
plt.ylim(-1.5,1)
plt.xlabel('Distance (Å)')
plt.ylabel('Potential (eV)')
plt.plot(r,e_LJ,color='black',label="Ref. Lennard-Jones potential")
plt.plot(r,e,'--',color='red',label="Fitted potential")
plt.legend()
plt.show()

### Using the "Sparse" option
We can use `"SearchMode": "Sparse"` to use a sparser grid in the search of inflection points. The resulotion in this sparse grid is given by `"SearchResolution"`.

In [ ]:
### Generate input.json file
import json

input={
    "General": {
        "Interface": "CCS",
        "FitForces": "True",
        "FitStresses": "True"
    },
    "TrainSet": "LJ.db",
    "Twobody": {
                "Ne-Ne": {
                        "Rcut": 5.0,
                        "Resolution": 0.02,
                        "SwType": "sw",
                        "ConstType" : "Mono",
                        "SearchMode": "Sparse",
                        "SearchResolution": 0.5
                }
        }
}
#SAVE TO FILE
with open('CCS_input.json', 'w') as f:
    json.dump(input, f, indent=8)

ccs_fit("CCS_input.json")

with open("CCS_params.json", "r") as f:
    CCS_params = json.load(f)

r=np.array(CCS_params["Two_body"]["Ne-Ne"]["r"])
e=CCS_params["Two_body"]["Ne-Ne"]["spl_a"]
e_LJ= 4 * ((1 / r) ** 12 - (1 / r) ** 6)
plt.xlim(0.5,3)
plt.ylim(-1.5,1)
plt.xlabel('Distance (Å)')
plt.ylabel('Potential (eV)')
plt.plot(r,e_LJ,color='black',label="Ref. Lennard-Jones potential")
plt.plot(r,e,'--',color='red',label="Fitted potential")
plt.legend()
plt.show()

***
***
##  7. Reset notebook

In [6]:
import os
import glob
import ipywidgets as widgets
from IPython.display import display, clear_output

# Define the directories
current_directory = os.getcwd()
calculated_data_directory = os.path.join(current_directory, 'CALCULATED_DATA')

# Function to remove files except for the specified file
def remove_files_except(directory, exception_file):
    for file_path in glob.glob(os.path.join(directory, '*')):
        if os.path.isfile(file_path) and not file_path.endswith(exception_file):
            os.remove(file_path)

# Cleanup function triggered by button click
def cleanup(b):
    with out:
        clear_output(wait=True)
        remove_files_except(current_directory, 'ipynb')
        remove_files_except(calculated_data_directory, 'p')
        print("Cleanup completed.")

# Function to cancel cleanup
def cancel_cleanup(b):
    with out:
        clear_output(wait=True)
        print("No changes made.")

# Create Yes/No buttons
button_yes = widgets.Button(description="Yes", button_style='danger')
button_no = widgets.Button(description="No", button_style='success')

button_yes.on_click(cleanup)
button_no.on_click(cancel_cleanup)

# Display prompt
print("Would you like to clean up?")
display(button_yes, button_no)

# Output area for messages
out = widgets.Output()
display(out)


Would you like to clean up?


Button(button_style='danger', description='Yes', style=ButtonStyle())

Button(button_style='success', description='No', style=ButtonStyle())

Output()